# Module 14 — Logging & Observability for AI Agents

> Part of the **"Develop & Deploy AI Agents on Azure with LangChain, Python and Foundry"** course.

Agents are non-deterministic. When something goes wrong in production — a tool returns garbage, the LLM loops, latency spikes — you need to **see exactly what happened, step by step**.

This module gives you three complementary observability layers.

## 🎯 Learning objectives

1. Stream **structured events** out of LangGraph using **callbacks** and `astream_events`.
2. Send agent traces to **Azure Monitor Application Insights** via **OpenTelemetry**.
3. Send the same traces to **LangSmith** for prompt-level debugging.
4. Define **KPIs** for agents (success rate, tool error rate, cost per task, p95 latency).
5. Set up an **Azure Monitor workbook** to track them.

## 🪜 The three layers

| Layer            | What it captures                            | Tool                                       |
| ---------------- | ------------------------------------------- | ------------------------------------------ |
| **Logs**         | Free-form text events                       | `print`, `logging`, ACA log stream         |
| **Metrics**      | Numeric counters / histograms               | Azure Monitor, Prometheus                  |
| **Traces**       | Spans linked into a tree (the agent loop!)  | **OpenTelemetry** → App Insights / LangSmith |

👉 For agents, **traces are king** — each LLM call and each tool call becomes a span; you can drill into them in App Insights' transaction view.

## 🛠 Install

In [ ]:
%pip install langchain langchain-openai langgraph langsmith \
             azure-monitor-opentelemetry opentelemetry-instrumentation-langchain

## 📺 Layer 1 — Stream events from LangGraph

Even before sending anything to a backend, you can already watch the agent **as it thinks**.

In [ ]:
import os
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

@tool
def square(x: int) -> int:
    """Return the square of x."""
    return x * x

model = ChatOpenAI(
    base_url=os.environ.get("BASE_URL", ""),
    api_key=os.environ.get("API_KEY", "EMPTY"),
    model=os.environ.get("MODEL", ""),
)
agent = create_agent(model=model, tools=[square])

async for event in agent.astream_events(
    {"messages": [{"role": "user", "content": "What is 9 squared?"}]},
    version="v2",
):
    if event["event"] in ("on_tool_start", "on_tool_end", "on_chat_model_end"):
        print(event["event"], "-", event.get("name"))

## ☁️ Layer 2 — Send traces to Azure Monitor (App Insights)

Get your App Insights **connection string** (Terraform output `appinsights_connection_string` if you provisioned one, or copy it from the Azure portal) and set it as an environment variable.

Then a **single line** enables auto-instrumentation for LangChain.

In [ ]:
import os
from azure.monitor.opentelemetry import configure_azure_monitor

# os.environ["APPLICATIONINSIGHTS_CONNECTION_STRING"] = "<your-connection-string>"

configure_azure_monitor(
    enable_live_metrics=True,
    instrumentation_options={
        "azure_sdk": {"enabled": True},
    },
)

from opentelemetry.instrumentation.langchain import LangchainInstrumentor
LangchainInstrumentor().instrument()

print("✅ App Insights configured. Every LangChain run from now on appears in the Transaction Search blade.")

## 🧪 Layer 3 — Send the same traces to LangSmith

LangSmith gives a **prompt-aware** UI that App Insights doesn't have (diff prompts, replay runs, evaluate). Setup is just three environment variables.

In [ ]:
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = "<lsv2_pt_...>"
# os.environ["LANGSMITH_PROJECT"] = "course-agent-demo"

result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "What is 12 squared?"}]}
)
print(result["messages"][-1].content)

## 📊 Agent KPIs to track in production

| Metric                 | Why it matters                                                      |
| ---------------------- | ------------------------------------------------------------------- |
| **Task success rate**  | Did the user actually get what they asked for? (LLM-judge or human) |
| **Steps per task**     | More steps = more cost. Outliers = LLM looping.                     |
| **Tool error rate**    | A broken tool = a broken agent.                                     |
| **Tokens per task**    | Direct $ cost.                                                       |
| **p95 latency**        | User experience.                                                    |
| **Refusal rate**       | Safety filter being too aggressive?                                  |

Use Azure Monitor **Workbooks** to chart these from the OTel data emitted above.

## 🧯 Production checklist

- [ ] App Insights connection string set as an ACA secret (never in code).
- [ ] PII scrubbing on log content (an OTel **span processor** is the right place).
- [ ] Sampling configured (`OTEL_TRACES_SAMPLER=parentbased_traceidratio`).
- [ ] Alert on **tool error rate > 5 %** for 10 min.
- [ ] Alert on **token cost** spike (cost guard).
- [ ] Workbook bookmarked & shared with the team.

You now have **eyes** on your agents.